# Forest Model — Trail Video Inference

Run a `.pt` model on an `.mp4` trail video and download the annotated result.

**Recommended:** `Runtime > Change runtime type > GPU` before running.

Assumes an Ultralytics YOLO `.pt` checkpoint (YOLOv8/YOLO11). If your model is a raw `torch.save` state dict from a different framework, see the alternate cell at the bottom.

## Step 1: Install dependencies

In [ ]:
!pip -q install ultralytics

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## Step 2: Upload your model and video

Upload the `.pt` weights first, then the `.mp4` trail video.

In [ ]:
from google.colab import files
import os

print('Upload your .pt model file:')
model_upload = files.upload()
model_path = next(iter(model_upload))
print(f'Model: {model_path} ({os.path.getsize(model_path)/1e6:.1f} MB)')

print('\nUpload your .mp4 trail video:')
video_upload = files.upload()
video_path = next(iter(video_upload))
print(f'Video: {video_path} ({os.path.getsize(video_path)/1e6:.1f} MB)')

## Step 3: Load model and inspect classes

In [ ]:
from ultralytics import YOLO

model = YOLO(model_path)
print('Task:    ', model.task)
print('Classes: ', model.names)

## Step 4: Run inference on the video

Tweak `conf` (confidence threshold) and `imgsz` to trade speed for accuracy. `stream=True` keeps memory low for long videos.

In [ ]:
import cv2
from pathlib import Path

CONF = 0.25
IMGSZ = 640
OUT_PATH = 'annotated.mp4'

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f'{total} frames, {w}x{h} @ {fps:.1f}fps')

writer = cv2.VideoWriter(OUT_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

results = model.predict(
    source=video_path,
    conf=CONF,
    imgsz=IMGSZ,
    stream=True,
    verbose=False,
)

detections_per_frame = []
for i, r in enumerate(results):
    writer.write(r.plot())
    detections_per_frame.append(len(r.boxes) if r.boxes is not None else 0)
    if i % 30 == 0:
        print(f'  frame {i}/{total}  detections={detections_per_frame[-1]}')

writer.release()
print(f'\nWrote {OUT_PATH} ({os.path.getsize(OUT_PATH)/1e6:.1f} MB)')
print(f'Avg detections/frame: {sum(detections_per_frame)/max(len(detections_per_frame),1):.2f}')

## Step 5: Download the annotated video

Re-encodes to H.264 (the `mp4v` codec OpenCV writes doesn't play in every viewer) and downloads.

In [ ]:
from google.colab import files

# Re-encode to H.264 so the downloaded file plays in any standard player.
!ffmpeg -y -loglevel error -i annotated.mp4 -vcodec libx264 -crf 23 -pix_fmt yuv420p annotated_h264.mp4

files.download('annotated_h264.mp4')

---
## Alternate: per-class detection counts

Useful if your model detects multiple forest/trail features (rocks, roots, obstacles, etc.).

In [ ]:
from collections import Counter

counts = Counter()
for r in model.predict(source=video_path, conf=CONF, imgsz=IMGSZ, stream=True, verbose=False):
    if r.boxes is None:
        continue
    for c in r.boxes.cls.tolist():
        counts[model.names[int(c)]] += 1

print('Total detections by class across the video:')
for name, n in counts.most_common():
    print(f'  {name:20s} {n}')

---
## If your `.pt` is NOT an Ultralytics YOLO checkpoint

A raw `torch.save(model.state_dict(), ...)` file needs the original model class to load. Replace the cell below with your model's architecture, then run inference frame-by-frame.

In [ ]:
# import torch
# from your_module import YourModel          # <-- your architecture
#
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# model = YourModel()
# model.load_state_dict(torch.load(model_path, map_location=device))
# model.eval().to(device)
#
# cap = cv2.VideoCapture(video_path)
# while True:
#     ok, frame = cap.read()
#     if not ok: break
#     # ... preprocess frame -> tensor, run model(tensor), draw outputs ...
# cap.release()